# Drive mounting

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Extraction from Google drive which was donwloaded by the kaggle link & Verification

In [2]:
import os

# paths
drive_zip_path = '/content/drive/MyDrive/ANTLINGS_Drone_CV/dataset/archive.zip'
local_extract_path = '/content/visdrone_dataset'

# local folder
os.makedirs(local_extract_path, exist_ok=True)

# Unzip dataset
print("Extracting dataset to local Colab storage")
!unzip -q "{drive_zip_path}" -d "{local_extract_path}"
print("Extraction complete!")

Extracting dataset to local Colab storage
Extraction complete!


In [4]:
!ls -lh "{local_extract_path}"

total 4.0K
drwxr-xr-x 6 root root 4.0K May 14 00:49 VisDrone_Dataset


# YAML Configuration & ultralytics installation

In [6]:
!find /content/visdrone_dataset -maxdepth 2 -type d

print("\n--- Searching for YAML Files ---")
!find /content/visdrone_dataset -name "*.yaml"

True/content/visdrone_dataset
/content/visdrone_dataset/VisDrone_Dataset
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-challenge
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev

--- Searching for YAML Files ---
/content/visdrone_dataset/VisDrone_Dataset/visdrone.yaml


In [7]:
import yaml

old_yaml_path = '/content/visdrone_dataset/VisDrone_Dataset/visdrone.yaml'
new_yaml_path = '/content/ants_visdrone.yaml'

try:
    # the original YAML provided by the Kaggle author
    with open(old_yaml_path, 'r') as f:
        data = yaml.safe_load(f)


    data['train'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images'
    data['val'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val/images'
    data['test'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images'

    # new production YAML
    with open(new_yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

    print(f"Production YAML successfully created at: {new_yaml_path}")
    print("\n--- YAML Contents ---")


    with open(new_yaml_path, 'r') as f:
        print(f.read())

except Exception as e:
    print(f"Error: {e}")

Production YAML successfully created at: /content/ants_visdrone.yaml

--- YAML Contents ---
path: ./VisDrone_Dataset
train: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images
val: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val/images
test: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images
nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor



In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00


# Visual Sanity Check  with Grid and Names

In [10]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics.utils.plotting import Annotator

# 1. Map our class IDs to the actual names from your YAML
class_names = {
    0: "pedestrian", 1: "people", 2: "bicycle", 3: "car",
    4: "van", 5: "truck", 6: "tricycle", 7: "awning-tricycle",
    8: "bus", 9: "motor"
}

# 2. Grab 4 random images to create a grid
train_images = glob.glob('/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images/*.jpg')
random_images = random.sample(train_images, 4)

plt.figure(figsize=(18, 14))

# 3. Loop through and plot them
for i, img_path in enumerate(random_images):
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    annotator = Annotator(img, line_width=2) # Thinner lines for tiny objects

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                cls_id, x, y, w, h = map(float, line.split())

                # Fetch the string name, fallback to ID if something is weird
                cls_name = class_names.get(int(cls_id), f"Unknown({int(cls_id)})")

                # YOLO formats are normalized (0 to 1). Convert back to pixels.
                ih, iw, _ = img.shape
                x1 = int((x - w/2) * iw)
                y1 = int((y - h/2) * ih)
                x2 = int((x + w/2) * iw)
                y2 = int((y + h/2) * ih)

                # Plot with the actual name
                annotator.box_label([x1, y1, x2, y2], label=cls_name)

    plt.subplot(2, 2, i+1)
    plt.imshow(annotator.result())
    plt.axis('off')
    plt.title(f"Sample {i+1}: {os.path.basename(img_path)}")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# Initializing Training

In [4]:
!pip install wandb -qU
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 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


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hijbullah (hijbullah-iubat-international-university-of-business-agr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

 ··········


In [ ]:
from ultralytics import YOLO

# Load the YOLOv26 Medium weights for maximum accuracy
model = YOLO('yolo26m.pt')

# training loop
results = model.train(
    data='/content/ants_visdrone.yaml',
    epochs=50,                           # 50 is a solid baseline for VisDrone
    imgsz=640,                           # Standard resolution
    batch=8,                             #  8 to prevent T4 OOM errors
    device=0,                            # to use the T4 GPU
    project='/content/drive/MyDrive/ANTLINGS_Drone_CV/runs', # Save straight to Drive
    name='yolo26m_visdrone_final',       # Folder name for this high-accuracy run
    save=True,                           # Save weights securely
    cache=False                          # False saves system RAM
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ants_visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_visdrone_final-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 